# CDT-III 2-Stage VCE: Quantitative Evaluation (NB1)
## RNA & Protein Per-Gene Evaluation

**Purpose**: Per-gene RNA and per-protein evaluation. Numerical foundation for the paper.

**Outputs**: `evaluation_results.json`, per-gene/per-protein CSV, summary figure

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Dict, Optional, Tuple, List
import numpy as np
import h5py
import pandas as pd
from pathlib import Path
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
import json
import gc

print(f"PyTorch: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Mounted at /content/drive
PyTorch: 2.10.0+cu128
Device: cuda


In [2]:
# ============================================================
# Path Configuration
# ============================================================
DRIVE_BASE = Path("/content/drive/MyDrive/cdt_data")
OUTPUT_BASE = Path("/content/drive/MyDrive/cdt_outputs/morris_2stage_vce_v2")

CELLLEVEL_WITH_ADT_PATH = DRIVE_BASE / "morris_celllevel_effects_with_adt.h5"
TSS_ENFORMER_PATH = DRIVE_BASE / "morris_28genes_enformer.h5"
BEST_MODEL_PATH = OUTPUT_BASE / "cdt_2stage_vce_best.pt"
ADT_MAPPING_PATH = "/content/drive/MyDrive/cdt_data/adt_mapping.csv"

FIGURES_DIR = OUTPUT_BASE / "evaluation_figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("File check:")
for name, path in [
    ("Cell-level + ADT", CELLLEVEL_WITH_ADT_PATH),
    ("TSS Enformer", TSS_ENFORMER_PATH),
    ("Best model", BEST_MODEL_PATH),
]:
    exists = path.exists()
    print(f"  {'✓' if exists else '✗'} {name}: {path}")

File check:
  ✓ Cell-level + ADT: /content/drive/MyDrive/cdt_data/morris_celllevel_effects_with_adt.h5
  ✓ TSS Enformer: /content/drive/MyDrive/cdt_data/morris_28genes_enformer.h5
  ✗ Best model: /content/drive/MyDrive/cdt_outputs/morris_2stage_vce_v2/cdt_2stage_vce_v2_best.pt


In [3]:
# ============================================================
# CDT-III 2-Stage VCE Model Definition
# (Copy from CDT_Morris_2StageVCE_v2_Training.ipynb cells 12-16)
# ============================================================

@dataclass
class CDT2StageVCEConfig:
    dna_dim: int = 3072
    dna_seq_len: int = 896
    n_genes: int = 2361
    n_proteins: int = 189
    hidden_dim: int = 512
    nhead: int = 8
    dropout: float = 0.3
    protein_dropout: float = 0.5
    dna_self_attn_layers: int = 2
    rna_self_attn_layers: int = 1
    protein_self_attn_layers: int = 1


class RawExpressionEncoder(nn.Module):
    def __init__(self, n_features, hidden_dim, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.hidden_dim = hidden_dim
        self.feature_embedding = nn.Embedding(n_features, hidden_dim)
        self.expr_projector = nn.Sequential(
            nn.Linear(1, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout))
        self.combine = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.Dropout(dropout))

    def forward(self, expression):
        B = expression.size(0)
        feat_ids = torch.arange(self.n_features, device=expression.device)
        feat_emb = self.feature_embedding(feat_ids).unsqueeze(0).expand(B, -1, -1)
        expr_emb = self.expr_projector(expression.unsqueeze(-1))
        return self.combine(torch.cat([feat_emb, expr_emb], dim=-1))


class SequenceProjector(nn.Module):
    def __init__(self, input_dim, output_dim, dropout=0.1):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
        self.norm = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.norm(self.linear(x)))


class FlashSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, nhead=8, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead
        self.dropout_p = dropout
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, return_attn=False):
        B, S, _ = x.shape
        Q = self.q_proj(x).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        if return_attn:
            scale = self.head_dim ** -0.5
            attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * scale
            attn_weights = F.softmax(attn_weights, dim=-1)
            attn_out = torch.matmul(attn_weights, V)
        else:
            attn_out = F.scaled_dot_product_attention(
                Q, K, V, dropout_p=self.dropout_p if self.training else 0.0)
            attn_weights = None
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, S, self.d_model)
        attn_out = self.out_proj(attn_out)
        x = self.norm1(x + self.dropout(attn_out))
        x = self.norm2(x + self.ffn(x))
        if return_attn:
            return x, attn_weights
        return x


class FlashCrossAttentionBlock(nn.Module):
    def __init__(self, d_model, nhead=8, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead
        self.dropout_p = dropout
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key_value, return_attn=False):
        B, QL, _ = query.shape
        KL = key_value.shape[1]
        Q = self.q_proj(query).view(B, QL, self.nhead, self.head_dim).transpose(1, 2)
        K = self.k_proj(key_value).view(B, KL, self.nhead, self.head_dim).transpose(1, 2)
        V = self.v_proj(key_value).view(B, KL, self.nhead, self.head_dim).transpose(1, 2)
        if return_attn:
            scale = self.head_dim ** -0.5
            attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * scale
            attn_weights = F.softmax(attn_weights, dim=-1)
            attn_out = torch.matmul(attn_weights, V)
        else:
            attn_out = F.scaled_dot_product_attention(
                Q, K, V, dropout_p=self.dropout_p if self.training else 0.0)
            attn_weights = None
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, QL, self.d_model)
        attn_out = self.out_proj(attn_out)
        x = self.norm1(query + self.dropout(attn_out))
        x = self.norm2(x + self.ffn(x))
        if return_attn:
            return x, attn_weights
        return x


class VirtualCellEmbedderDNARNA(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = 4
        self.head_dim = d_model // self.nhead
        self.dna_query = nn.Parameter(torch.randn(1, 1, d_model))
        self.rna_query = nn.Parameter(torch.randn(1, 1, d_model))
        self.dna_q_proj = nn.Linear(d_model, d_model)
        self.dna_k_proj = nn.Linear(d_model, d_model)
        self.dna_v_proj = nn.Linear(d_model, d_model)
        self.dna_out_proj = nn.Linear(d_model, d_model)
        self.rna_q_proj = nn.Linear(d_model, d_model)
        self.rna_k_proj = nn.Linear(d_model, d_model)
        self.rna_v_proj = nn.Linear(d_model, d_model)
        self.rna_out_proj = nn.Linear(d_model, d_model)
        self.fusion = nn.Sequential(
            nn.Linear(d_model * 2, d_model * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model), nn.LayerNorm(d_model))

    def _attention_pool(self, query, key_value, q_proj, k_proj, v_proj, out_proj):
        B, S = key_value.size(0), key_value.size(1)
        query = query.expand(B, -1, -1)
        Q = q_proj(query).view(B, 1, self.nhead, self.head_dim).transpose(1, 2)
        K = k_proj(key_value).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        V = v_proj(key_value).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        attn_out = F.scaled_dot_product_attention(Q, K, V)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, 1, self.d_model)
        return out_proj(attn_out).squeeze(1)

    def forward(self, dna_encoded, rna_encoded):
        dna_pooled = self._attention_pool(
            self.dna_query, dna_encoded,
            self.dna_q_proj, self.dna_k_proj, self.dna_v_proj, self.dna_out_proj)
        rna_pooled = self._attention_pool(
            self.rna_query, rna_encoded,
            self.rna_q_proj, self.rna_k_proj, self.rna_v_proj, self.rna_out_proj)
        return self.fusion(torch.cat([dna_pooled, rna_pooled], dim=-1))


class VirtualCellEmbedderProtein(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = 4
        self.head_dim = d_model // self.nhead
        self.protein_query = nn.Parameter(torch.randn(1, 1, d_model))
        self.protein_q_proj = nn.Linear(d_model, d_model)
        self.protein_k_proj = nn.Linear(d_model, d_model)
        self.protein_v_proj = nn.Linear(d_model, d_model)
        self.protein_out_proj = nn.Linear(d_model, d_model)
        self.fusion = nn.Sequential(
            nn.Linear(d_model * 2, d_model * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model), nn.LayerNorm(d_model))

    def _attention_pool(self, query, key_value, q_proj, k_proj, v_proj, out_proj):
        B, S = key_value.size(0), key_value.size(1)
        query = query.expand(B, -1, -1)
        Q = q_proj(query).view(B, 1, self.nhead, self.head_dim).transpose(1, 2)
        K = k_proj(key_value).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        V = v_proj(key_value).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        attn_out = F.scaled_dot_product_attention(Q, K, V)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, 1, self.d_model)
        return out_proj(attn_out).squeeze(1)

    def forward(self, cell_emb_rna, protein_encoded):
        protein_pooled = self._attention_pool(
            self.protein_query, protein_encoded,
            self.protein_q_proj, self.protein_k_proj,
            self.protein_v_proj, self.protein_out_proj)
        return self.fusion(torch.cat([cell_emb_rna, protein_pooled], dim=-1))


class CDTTrimodal2StageModel(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        if config is None:
            config = CDT2StageVCEConfig()
        self.config = config
        self.dna_projector = SequenceProjector(config.dna_dim, config.hidden_dim, config.dropout)
        self.dna_self_attn_layers = nn.ModuleList([
            FlashSelfAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
            for _ in range(config.dna_self_attn_layers)])
        self.rna_encoder = RawExpressionEncoder(config.n_genes, config.hidden_dim, config.dropout)
        self.rna_self_attn_layers = nn.ModuleList([
            FlashSelfAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
            for _ in range(config.rna_self_attn_layers)])
        self.dna_to_rna = FlashCrossAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
        self.vce_t = VirtualCellEmbedderDNARNA(config.hidden_dim, config.dropout)
        self.rna_task_layer = nn.Sequential(
            nn.Linear(config.hidden_dim, config.hidden_dim), nn.GELU(),
            nn.Dropout(config.dropout), nn.Linear(config.hidden_dim, config.n_genes))
        self.protein_encoder = RawExpressionEncoder(
            config.n_proteins, config.hidden_dim, config.protein_dropout)
        self.protein_self_attn_layers = nn.ModuleList([
            FlashSelfAttentionBlock(config.hidden_dim, config.nhead, config.protein_dropout)
            for _ in range(config.protein_self_attn_layers)])
        self.rna_to_protein = FlashCrossAttentionBlock(
            config.hidden_dim, config.nhead, config.protein_dropout)
        self.vce_p = VirtualCellEmbedderProtein(config.hidden_dim, config.protein_dropout)
        self.protein_task_layer = nn.Sequential(
            nn.Linear(config.hidden_dim, config.hidden_dim), nn.GELU(),
            nn.Dropout(config.protein_dropout),
            nn.Linear(config.hidden_dim, config.n_proteins))

    def forward(self, dna_emb, rna_expr, protein_expr, return_attention=False):
        attn_maps = {} if return_attention else None
        dna = self.dna_projector(dna_emb)
        rna = self.rna_encoder(rna_expr)
        protein = self.protein_encoder(protein_expr)
        for i, layer in enumerate(self.dna_self_attn_layers):
            if return_attention:
                dna, attn_w = layer(dna, return_attn=True)
                attn_maps[f'dna_self_attn_{i}'] = attn_w.detach().cpu()
            else:
                dna = layer(dna)
        for i, layer in enumerate(self.rna_self_attn_layers):
            if return_attention:
                rna, attn_w = layer(rna, return_attn=True)
                attn_maps[f'rna_self_attn_{i}'] = attn_w.detach().cpu()
            else:
                rna = layer(rna)
        for i, layer in enumerate(self.protein_self_attn_layers):
            if return_attention:
                protein, attn_w = layer(protein, return_attn=True)
                attn_maps[f'protein_self_attn_{i}'] = attn_w.detach().cpu()
            else:
                protein = layer(protein)
        if return_attention:
            rna, attn_w = self.dna_to_rna(query=rna, key_value=dna, return_attn=True)
            attn_maps['dna_to_rna_cross'] = attn_w.detach().cpu()
        else:
            rna = self.dna_to_rna(query=rna, key_value=dna)
        if return_attention:
            protein, attn_w = self.rna_to_protein(query=protein, key_value=rna, return_attn=True)
            attn_maps['rna_to_protein_cross'] = attn_w.detach().cpu()
        else:
            protein = self.rna_to_protein(query=protein, key_value=rna)
        cell_emb_rna = self.vce_t(dna, rna)
        rna_pred = self.rna_task_layer(cell_emb_rna)
        cell_emb_protein = self.vce_p(cell_emb_rna, protein)
        protein_pred = self.protein_task_layer(cell_emb_protein)
        if return_attention:
            return rna_pred, protein_pred, attn_maps
        return rna_pred, protein_pred

    def get_num_params(self, trainable_only=False):
        if trainable_only:
            return sum(p.numel() for p in self.parameters() if p.requires_grad)
        return sum(p.numel() for p in self.parameters())

print("CDTTrimodal2StageModel defined.")

CDTTrimodal2StageModel defined.


In [7]:
# ============================================================
# Load Data
# ============================================================
print("Loading integrated RNA + ADT data...")
with h5py.File(CELLLEVEL_WITH_ADT_PATH, 'r') as f:
    rna_log2fc = f['log2fc'][:]
    cell_expr = f['cell_expr'][:]
    target_gene_idx = f['target_gene_idx'][:]
    cdt_genes = [g.decode() if isinstance(g, bytes) else g for g in f['gene_names_cdt'][:]]
    target_gene_names = [g.decode() if isinstance(g, bytes) else g for g in f['target_gene_names'][:]]
    ntc_mean_expr = f['ntc_mean_expr'][:]
    train_genes = [g.decode() if isinstance(g, bytes) else g for g in f['train_genes'][:]]
    val_genes = [g.decode() if isinstance(g, bytes) else g for g in f['val_genes'][:]]
    protein_effect = f['protein_effect'][:]
    protein_dsb = f['protein_dsb'][:]
    protein_names = [g.decode() if isinstance(g, bytes) else g for g in f['protein_names'][:]]
    n_cells = f.attrs['n_cells']
    n_genes = f.attrs['n_genes']
    n_proteins = f.attrs['n_proteins']

    # Check if 'protein_matched_mask' exists, otherwise create a default all-True mask
    if 'protein_matched_mask' in f:
        protein_matched_mask = f['protein_matched_mask'][:]
    else:
        print("Warning: 'protein_matched_mask' not found in HDF5. Creating an all-True mask.")
        protein_matched_mask = np.ones(n_cells, dtype=bool)

print(f"RNA: {n_cells} cells, {n_genes} genes, log2FC {rna_log2fc.shape}")
print(f"Protein: {n_proteins} proteins, effect {protein_effect.shape}")
print(f"Cells with protein: {protein_matched_mask.sum()}/{n_cells}")
print(f"Train genes ({len(train_genes)}): {train_genes[:5]}...")
print(f"Val genes ({len(val_genes)}): {val_genes}")

# Enformer
print("\nLoading Enformer embeddings...")
with h5py.File(TSS_ENFORMER_PATH, 'r') as f:
    enformer_emb = f['embeddings'][:]
    enformer_genes = [g.decode() if isinstance(g, bytes) else g for g in f['gene_names'][:]]
print(f"Enformer: {enformer_emb.shape}")

target_to_enformer = {}
for i, gene in enumerate(target_gene_names):
    if gene in enformer_genes:
        target_to_enformer[i] = enformer_genes.index(gene)
print(f"TSS matched: {len(target_to_enformer)}/{len(target_gene_names)}")

# ADT mapping
try:
    adt_mapping = pd.read_csv(ADT_MAPPING_PATH)
    adt_mapping = adt_mapping[~adt_mapping['morris_adt_id'].str.startswith('ADT-CTL')]
    adt_to_gene = dict(zip(adt_mapping['morris_adt_id'], adt_mapping['gene_symbol']))
    adt_to_protein = dict(zip(adt_mapping['morris_adt_id'], adt_mapping['protein_name']))
    print(f"ADT mapping: {len(adt_to_gene)} proteins mapped")
except:
    print("ADT mapping not found, using protein_names directly")
    adt_to_gene = {}
    adt_to_protein = {}

# Index maps
gene_to_cdt_idx = {g: i for i, g in enumerate(cdt_genes)}
gene_name_to_target_idx = {n: i for i, n in enumerate(target_gene_names)}

# ADT screening: expressed proteins
mean_dsb = protein_dsb.mean(axis=0)
std_dsb = protein_dsb.std(axis=0)
expressed_mask = (mean_dsb > 0.5) & (std_dsb > 0.1)
expressed_indices = np.where(expressed_mask)[0]
n_expressed = expressed_mask.sum()
print(f"\nExpressed proteins: {n_expressed}/{n_proteins}")

Loading integrated RNA + ADT data...
RNA: 8250 cells, 2361 genes, log2FC (8250, 2361)
Protein: 189 proteins, effect (8250, 189)
Cells with protein: 8250/8250
Train genes (22): ['B2M', 'CD19', 'CD40', 'CD58', 'CD69']...
Val genes (5): ['CD44', 'CD52', 'GFI1B', 'TFRC', 'TNFSF9']

Loading Enformer embeddings...
Enformer: (28, 896, 3072)
TSS matched: 27/27
ADT mapping not found, using protein_names directly

Expressed proteins: 65/189


In [9]:
import os
output_dir = "/content/drive/MyDrive/cdt_outputs/morris_2stage_vce_v2/"
if os.path.exists(output_dir):
    for f in sorted(os.listdir(output_dir)):
        size = os.path.getsize(os.path.join(output_dir, f))
        print(f"  {f} ({size/1e6:.1f} MB)")
else:
    print("Directory not found!")
    # Check parent
    parent = "/content/drive/MyDrive/cdt_outputs/"
    if os.path.exists(parent):
        print("Available dirs:")
        for d in sorted(os.listdir(parent)):
            print(f"  {d}")


  adt_screening.png (0.1 MB)
  cdt_2stage_vce_best.pt (124.0 MB)
  checkpoint_latest.pt (372.1 MB)
  evaluation_figures (0.0 MB)
  results_2stage_vce_20260317_063355.json (0.0 MB)
  training_history_2stage.png (0.3 MB)


In [8]:
# ============================================================
# Load Best Model Checkpoint
# ============================================================
print("Loading best model checkpoint...")
config = CDT2StageVCEConfig(n_genes=len(cdt_genes), n_proteins=len(protein_names))
model = CDTTrimodal2StageModel(config).to(device)

ckpt = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)
if 'model_state_dict' in ckpt:
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded from checkpoint with keys: {list(ckpt.keys())}")
    if 'val_rna_r' in ckpt:
        print(f"  Val RNA r: {ckpt['val_rna_r']:.4f}")
    if 'val_protein_r' in ckpt:
        print(f"  Val Protein r: {ckpt['val_protein_r']:.4f}")
    if 'epoch' in ckpt:
        print(f"  Epoch: {ckpt['epoch']}")
else:
    model.load_state_dict(ckpt)
    print("Loaded raw state dict")

model.eval()
print(f"Parameters: {model.get_num_params():,}")

Loading best model checkpoint...


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/cdt_outputs/morris_2stage_vce_v2/cdt_2stage_vce_v2_best.pt'

In [ ]:
# ============================================================
# Per-Gene RNA + Per-Protein Predictions (Val Set)
# ============================================================

def predict_gene(model, gene_name, cell_indices, cell_expr, protein_dsb,
                 rna_log2fc, protein_effect, enformer_emb,
                 target_gene_idx, target_to_enformer, device):
    """Run predictions for all cells of a given target gene."""
    rna_preds, rna_targets = [], []
    prot_preds, prot_targets = [], []

    for ci in cell_indices:
        enf_idx = target_to_enformer[target_gene_idx[ci]]
        dna_t = torch.from_numpy(enformer_emb[enf_idx:enf_idx+1].astype(np.float32)).to(device)
        rna_t = torch.from_numpy(cell_expr[ci:ci+1].astype(np.float32)).to(device)
        prot_t = torch.from_numpy(protein_dsb[ci:ci+1].astype(np.float32)).to(device)

        with torch.no_grad():
            rna_pred, prot_pred = model(dna_t, rna_t, prot_t)

        rna_preds.append(rna_pred.cpu().numpy().flatten())
        rna_targets.append(rna_log2fc[ci])
        prot_preds.append(prot_pred.cpu().numpy().flatten())
        prot_targets.append(protein_effect[ci])

    return (np.array(rna_preds), np.array(rna_targets),
            np.array(prot_preds), np.array(prot_targets))


# Collect results per val gene
results = {}
for gene_name in val_genes:
    if gene_name not in gene_name_to_target_idx:
        print(f"  {gene_name}: not in target genes, skipping")
        continue
    g_target_idx = gene_name_to_target_idx[gene_name]
    if g_target_idx not in target_to_enformer:
        print(f"  {gene_name}: no Enformer embedding, skipping")
        continue

    gene_cells = [i for i in range(n_cells)
                  if target_gene_idx[i] == g_target_idx
                  and protein_matched_mask[i]]

    if len(gene_cells) < 5:
        print(f"  {gene_name}: only {len(gene_cells)} cells, skipping")
        continue

    rna_preds, rna_targets, prot_preds, prot_targets = predict_gene(
        model, gene_name, gene_cells, cell_expr, protein_dsb,
        rna_log2fc, protein_effect, enformer_emb,
        target_gene_idx, target_to_enformer, device)

    # Per-gene RNA correlations
    mean_rna_pred = rna_preds.mean(axis=0)
    mean_rna_target = rna_targets.mean(axis=0)
    rna_r_all = pearsonr(rna_preds.flatten(), rna_targets.flatten())[0]
    rna_r_mean = pearsonr(mean_rna_pred, mean_rna_target)[0]

    # Self-KD: target gene's own prediction
    g_cdt_idx = gene_to_cdt_idx.get(gene_name)
    self_kd_r = float('nan')
    if g_cdt_idx is not None:
        self_kd_r = pearsonr(rna_preds[:, g_cdt_idx], rna_targets[:, g_cdt_idx])[0]

    # Per-protein correlations
    mean_prot_pred = prot_preds.mean(axis=0)
    mean_prot_target = prot_targets.mean(axis=0)
    prot_r_all = pearsonr(prot_preds.flatten(), prot_targets.flatten())[0]
    prot_r_mean = pearsonr(mean_prot_pred, mean_prot_target)[0]

    # Expressed proteins only
    prot_r_mean_expr = pearsonr(
        mean_prot_pred[expressed_mask], mean_prot_target[expressed_mask])[0]

    results[gene_name] = {
        'n_cells': len(gene_cells),
        'rna_r_all': rna_r_all,
        'rna_r_mean': rna_r_mean,
        'self_kd_r': self_kd_r,
        'prot_r_all': prot_r_all,
        'prot_r_mean': prot_r_mean,
        'prot_r_mean_expressed': prot_r_mean_expr,
        'rna_preds': rna_preds,
        'rna_targets': rna_targets,
        'prot_preds': prot_preds,
        'prot_targets': prot_targets,
    }
    print(f"  {gene_name}: {len(gene_cells)} cells | "
          f"RNA r(all)={rna_r_all:.4f} r(mean)={rna_r_mean:.4f} self-KD={self_kd_r:.4f} | "
          f"Prot r(mean)={prot_r_mean:.4f} r(expr)={prot_r_mean_expr:.4f}")
    gc.collect()

print(f"\nProcessed {len(results)} val genes.")

In [ ]:
# ============================================================
# Per-Protein Evaluation (across all val genes)
# ============================================================

# Stack all val gene predictions
all_prot_preds = np.concatenate([r['prot_preds'] for r in results.values()], axis=0)
all_prot_targets = np.concatenate([r['prot_targets'] for r in results.values()], axis=0)
mean_all_prot_pred = all_prot_preds.mean(axis=0)
mean_all_prot_target = all_prot_targets.mean(axis=0)

per_protein_results = []
for i in range(n_proteins):
    r_all = pearsonr(all_prot_preds[:, i], all_prot_targets[:, i])[0]
    r_mean_vals = []
    for gene_name, res in results.items():
        mp = res['prot_preds'].mean(axis=0)[i]
        mt = res['prot_targets'].mean(axis=0)[i]
        r_mean_vals.append((mp, mt))
    preds_mean = [x[0] for x in r_mean_vals]
    targets_mean = [x[1] for x in r_mean_vals]
    r_mean = pearsonr(preds_mean, targets_mean)[0] if len(r_mean_vals) >= 3 else float('nan')

    prot_label = adt_to_protein.get(protein_names[i], protein_names[i])
    gene_label = adt_to_gene.get(protein_names[i], '')

    per_protein_results.append({
        'protein_idx': i,
        'protein_id': protein_names[i],
        'protein_name': prot_label,
        'gene_symbol': gene_label,
        'expressed': bool(expressed_mask[i]),
        'mean_dsb': float(mean_dsb[i]),
        'r_all': r_all,
        'r_cross_gene_mean': r_mean,
    })

prot_df = pd.DataFrame(per_protein_results)
prot_df_expr = prot_df[prot_df['expressed']].sort_values('r_all', ascending=False)

print(f"Per-Protein Results (expressed only, n={len(prot_df_expr)}):")
print(prot_df_expr[['protein_name', 'gene_symbol', 'mean_dsb', 'r_all']].to_string(index=False))
print(f"\nMean r(all) expressed: {prot_df_expr['r_all'].mean():.4f}")
print(f"Median r(all) expressed: {prot_df_expr['r_all'].median():.4f}")

In [ ]:
# ============================================================
# Figure 2 (NeurIPS): Prediction Performance — 3 Panels
# ============================================================
# (A) Single-stage vs Two-stage VCE (RNA r by architecture)
# (B) CDT-II vs CDT-III Per-gene RNA r
# (C) Per-gene Protein predicted vs actual scatter (65 expressed)
#
# Panels A, B use hardcoded values from the paper tables.
# Panel C uses results from Cell 7 (pseudo-bulk per gene).

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from scipy.stats import pearsonr

FIGURES_DIR = Path(output_dir) / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Also save to local figures/neurips/
LOCAL_FIG = Path("figures/neurips")
LOCAL_FIG.mkdir(parents=True, exist_ok=True)

# ---- Data for Panel A (Table 5 / Ablation) ----
ablation_labels = [
    'Exp A (λ=1.0)',
    'Exp B (λ=0.11)',
    'Exp C (λ=0.05)',
    'Exp D (λ=0.01)',
    'Exp E (warmup)',
    'CDT-III (2-Stage)',
]
ablation_rna_r = [0.19, 0.24, 0.35, 0.37, 0.30, 0.64]

# ---- Data for Panel B (Table 1 / Prediction) ----
genes_b = ['GFI1B', 'TNFSF9', 'TFRC', 'CD44', 'CD52']
cdt2_r  = [0.851, 0.831, 0.824, 0.795, 0.719]
cdt3_r  = [0.885, 0.868, 0.854, 0.858, 0.748]

# ---- Figure ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=300)

# ======== Panel A: Ablation — Single-stage vs Two-stage ========
ax = axes[0]
y_pos = np.arange(len(ablation_labels))
colors_a = ['#90CAF9'] * 5 + ['#D32F2F']  # CDT-III in red
bars = ax.barh(y_pos, ablation_rna_r, color=colors_a, edgecolor='white', linewidth=0.5, height=0.6)
ax.set_yticks(y_pos)
ax.set_yticklabels(ablation_labels, fontsize=10)
ax.set_xlabel('RNA Pearson r (per-gene mean)', fontsize=11)
ax.set_title('A', fontsize=14, fontweight='bold', loc='left')
ax.set_xlim(0, 0.75)
ax.invert_yaxis()

# Value labels
for i, v in enumerate(ablation_rna_r):
    ax.text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=9,
            fontweight='bold' if i == 5 else 'normal',
            color='#D32F2F' if i == 5 else '#333')

# Bracket annotation
ax.annotate('Single-stage\n(joint loss)', xy=(0.02, 2), fontsize=8, color='#666',
            va='center', ha='left',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='#E3F2FD', alpha=0.5))

# ======== Panel B: CDT-II vs CDT-III per-gene RNA r ========
ax = axes[1]
y_pos_b = np.arange(len(genes_b))
bar_height = 0.35

bars_ii = ax.barh(y_pos_b + bar_height/2, cdt2_r, bar_height,
                   color='#64B5F6', label='CDT-II (RNA only)', edgecolor='white')
bars_iii = ax.barh(y_pos_b - bar_height/2, cdt3_r, bar_height,
                    color='#D32F2F', label='CDT-III (+ Protein)', edgecolor='white')

ax.set_yticks(y_pos_b)
ax.set_yticklabels([f'$\\it{{{g}}}$' for g in genes_b], fontsize=10)
ax.set_xlabel('RNA Pearson r (per-gene mean)', fontsize=11)
ax.set_title('B', fontsize=14, fontweight='bold', loc='left')
ax.set_xlim(0.65, 0.95)
ax.invert_yaxis()
ax.legend(fontsize=9, loc='lower right')

# Improvement annotations
for i in range(len(genes_b)):
    diff = cdt3_r[i] - cdt2_r[i]
    ax.text(cdt3_r[i] + 0.005, y_pos_b[i] - bar_height/2,
            f'+{diff:.3f}', va='center', fontsize=8, color='#D32F2F')

# ======== Panel C: Per-gene Protein scatter (65 expressed, pseudo-bulk) ========
ax = axes[2]

# Extract pseudo-bulk protein predictions per gene from results dict
# results[gene]['prot_preds'] is [n_cells, n_proteins] (cell-level)
# We need mean across cells → [n_proteins], then filter to expressed_mask
gene_prot_actual = []
gene_prot_pred = []
gene_labels_c = []

for gene_name, r in results.items():
    # Pseudo-bulk: mean across all cells for this gene
    mean_actual = r['prot_targets'].mean(axis=0)  # [n_proteins]
    mean_pred = r['prot_preds'].mean(axis=0)       # [n_proteins]

    # Filter to expressed proteins only
    actual_expr = mean_actual[expressed_mask]
    pred_expr = mean_pred[expressed_mask]

    for j in range(len(actual_expr)):
        gene_prot_actual.append(actual_expr[j])
        gene_prot_pred.append(pred_expr[j])
        gene_labels_c.append(gene_name)

gene_prot_actual = np.array(gene_prot_actual)
gene_prot_pred = np.array(gene_prot_pred)
gene_labels_c = np.array(gene_labels_c)

print(f"Panel C: {len(gene_prot_actual)} points ({len(results)} genes × {expressed_mask.sum()} proteins)")

# Color by gene
gene_colors = {'GFI1B': '#1976D2', 'TNFSF9': '#388E3C', 'TFRC': '#F57C00',
               'CD44': '#7B1FA2', 'CD52': '#D32F2F'}
for gene_name in results:
    mask = gene_labels_c == gene_name
    color = gene_colors.get(gene_name, '#999')
    ax.scatter(gene_prot_actual[mask], gene_prot_pred[mask],
               s=25, c=color, alpha=0.6, label=f'$\\it{{{gene_name}}}$',
               edgecolors='white', linewidths=0.3)

# Diagonal
lim = max(abs(gene_prot_actual).max(), abs(gene_prot_pred).max()) * 1.1
ax.plot([-lim, lim], [-lim, lim], 'k--', alpha=0.4, linewidth=1)
ax.axhline(0, color='gray', alpha=0.2)
ax.axvline(0, color='gray', alpha=0.2)

# Compute per-gene r and overall mean r
per_gene_r = {}
for gene_name in results:
    mask = gene_labels_c == gene_name
    r_val, _ = pearsonr(gene_prot_actual[mask], gene_prot_pred[mask])
    per_gene_r[gene_name] = r_val
    print(f"  {gene_name}: protein r = {r_val:.3f}")
mean_r = np.mean(list(per_gene_r.values()))
print(f"  Mean protein r = {mean_r:.3f}")

ax.set_xlabel('Actual Protein Effect (pseudo-bulk)', fontsize=11)
ax.set_ylabel('Predicted Protein Effect', fontsize=11)
ax.set_title('C', fontsize=14, fontweight='bold', loc='left')
ax.legend(fontsize=8, loc='lower right', framealpha=0.8)
ax.text(0.05, 0.95, f'Mean r = {mean_r:.3f}\n(65 expressed proteins × 5 genes)',
        transform=ax.transAxes, fontsize=9, va='top',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.7))

plt.tight_layout()
save_path = FIGURES_DIR / 'evaluation_summary.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(LOCAL_FIG / 'evaluation_summary.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f"Saved: {save_path}")
print(f"Saved: {LOCAL_FIG / 'evaluation_summary.png'}")

In [ ]:
# ============================================================
# Save Results
# ============================================================

# JSON summary
eval_summary = {
    'model': 'CDT-III 2-Stage VCE v2',
    'checkpoint': str(BEST_MODEL_PATH),
    'n_val_genes': len(results),
    'val_genes': list(results.keys()),
    'per_gene': {},
    'overall': {
        'rna_r_mean_avg': float(np.mean([r['rna_r_mean'] for r in results.values()])),
        'prot_r_mean_expr_avg': float(np.mean([r['prot_r_mean_expressed'] for r in results.values()])),
        'n_expressed_proteins': int(n_expressed),
    }
}

for gene_name, res in results.items():
    eval_summary['per_gene'][gene_name] = {
        'n_cells': res['n_cells'],
        'rna_r_all': float(res['rna_r_all']),
        'rna_r_mean': float(res['rna_r_mean']),
        'self_kd_r': float(res['self_kd_r']),
        'prot_r_all': float(res['prot_r_all']),
        'prot_r_mean': float(res['prot_r_mean']),
        'prot_r_mean_expressed': float(res['prot_r_mean_expressed']),
    }

with open(OUTPUT_BASE / 'evaluation_results.json', 'w') as f:
    json.dump(eval_summary, f, indent=2)

# Per-protein CSV
prot_df.to_csv(OUTPUT_BASE / 'per_protein_evaluation.csv', index=False)

# Per-gene CSV
gene_df = pd.DataFrame([
    {'gene': g, **{k: v for k, v in res.items()
                   if not isinstance(v, np.ndarray)}}
    for g, res in results.items()
])
gene_df.to_csv(OUTPUT_BASE / 'per_gene_evaluation.csv', index=False)

print("Saved:")
print(f"  {OUTPUT_BASE / 'evaluation_results.json'}")
print(f"  {OUTPUT_BASE / 'per_protein_evaluation.csv'}")
print(f"  {OUTPUT_BASE / 'per_gene_evaluation.csv'}")

print(f"\n{'='*60}")
print("CDT-III 2-Stage VCE Evaluation Complete")
print(f"{'='*60}")
print(f"RNA r(mean) avg: {eval_summary['overall']['rna_r_mean_avg']:.4f}")
print(f"Protein r(mean,expressed) avg: {eval_summary['overall']['prot_r_mean_expr_avg']:.4f}")